# Phase 3 - Step 1: Rhetorical Role data exploration

Goal: understand the RR data before designing the model.

- file format, documents, sentences, labels (CL and IT)
- data problems (length mismatches, unknown labels, empty sentences, duplicates)
- label imbalance (Macro-F1 gives the rare roles the same weight as Fact)
- sentence and document length in bpe32k tokens (decides how much context fits in 1536 tokens)
- sequence structure (do labels come in runs? where in the document is each role?)
  No GPU needed. Results go to experiments/phase3/p3_01_data.


In [1]:
import json, os, hashlib, collections
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Find the project root the same way as the Phase 1/2 notebooks: walk up until .gitignore
ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW_DIR = ROOT / "data" / "raw"
P3 = RAW_DIR / "Phase_3_RR"
OUT = ROOT / "experiments" / "phase3" / "p3_01_data"
OUT.mkdir(parents=True, exist_ok=True)
TOK_PATH = ROOT / "data" / "tokenizer" / "bpe32k.json"   # my Phase-1 tokenizer
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("project root:", ROOT)

project root: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project


c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Download Phase 3 files (skipped if already on disk)


In [2]:
if (P3 / "Train" / "label_vocab.json").exists():
    print("Phase 3 data already downloaded -> skipping download")
else:
    from huggingface_hub import snapshot_download
    snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset",
                      allow_patterns="Phase_3_RR/*", local_dir=str(RAW_DIR))
for p in sorted(P3.rglob("*")):
    if p.is_file():
        print(f"{str(p.relative_to(P3)):30s} {p.stat().st_size / 1e6:8.2f} MB")

Fetching 7 files: 100%|██████████| 7/7 [00:01<00:00,  4.87it/s]

Test\CL_test.json                  0.28 MB
Test\IT_test.json                  0.19 MB
Train\CL_dev.json                  0.34 MB
Train\CL_train.json                2.46 MB
Train\IT_dev.json                  0.17 MB
Train\IT_train.json                1.51 MB
Train\label_vocab.json             0.00 MB


## 2. Load files and look at the format

The loader accepts a list of documents or a dict {doc_id: document}, so it works whichever format the files use.


In [3]:
def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def to_doc_list(data):
    """List of documents, each a dict that has an 'id'."""
    if isinstance(data, list):
        return [dict(d) for d in data]
    if isinstance(data, dict):
        docs = []
        for k, v in data.items():
            d = dict(v) if isinstance(v, dict) else {"text": v}
            d.setdefault("id", k)
            docs.append(d)
        return docs
    raise TypeError(f"unexpected JSON type {type(data)}")

def parse_vocab(v):
    """Label list in id order, from a list, {label: id}, {id: label}, {'labels': [...]} or {'label2id': {...}}."""
    if isinstance(v, list):
        return list(v)
    if "label2id" in v:
        return parse_vocab(v["label2id"])
    lists = [x for x in v.values() if isinstance(x, list)]
    if lists and len(v) <= 2:
        return list(lists[0])
    if all(isinstance(x, int) for x in v.values()):
        return [k for k, _ in sorted(v.items(), key=lambda kv: kv[1])]
    return [v[k] for k in sorted(v, key=int)]

vocab_raw = read_json(P3 / "Train" / "label_vocab.json")
LABELS = parse_vocab(vocab_raw)
LABEL_SET = set(LABELS)
print("label_vocab.json raw:", vocab_raw)
print(f"{len(LABELS)} labels:", LABELS)

SPLIT_FILES = {
    "CL_train": P3 / "Train" / "CL_train.json", "CL_dev": P3 / "Train" / "CL_dev.json",
    "IT_train": P3 / "Train" / "IT_train.json", "IT_dev": P3 / "Train" / "IT_dev.json",
    "CL_test":  P3 / "Test" / "CL_test.json",   "IT_test": P3 / "Test" / "IT_test.json",
}
known = set(SPLIT_FILES.values()) | {P3 / "Train" / "label_vocab.json"}
for p in P3.rglob("*.json"):
    if p not in known:
        print("WARNING: extra file not loaded:", p.relative_to(P3))

SPLITS = {}
for name, path in SPLIT_FILES.items():
    if not path.exists():
        print(f"WARNING: {name} missing ({path.relative_to(ROOT)})")
        continue
    raw = read_json(path)
    docs = to_doc_list(raw)
    SPLITS[name] = docs
    first = docs[0]
    print(f"\n===== {name}: file type {type(raw).__name__}, {len(docs)} documents, keys {list(first.keys())}")
    assert isinstance(first["text"], list), "expected 'text' to be a list of sentences"
    for i, s in enumerate(first["text"][:3]):
        lab = first["labels"][i] if "labels" in first else "-"
        print(f"  [{lab:>22}] {s[:110]}")

label_vocab.json raw: {'Fact': 0, 'Issue': 1, 'ArgumentPetitioner': 2, 'ArgumentRespondent': 3, 'Statute': 4, 'Dissent': 5, 'PrecedentReliedUpon': 6, 'PrecedentNotReliedUpon': 7, 'PrecedentOverruled': 8, 'RulingByLowerCourt': 9, 'RatioOfTheDecision': 10, 'RulingByPresentCourt': 11, 'None': 12}
13 labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']

===== CL_train: file type list, 40 documents, keys ['id', 'text', 'labels']
  [                  Fact] Consequent upon the repeal of the MRTP Act this complaint has been received by transfer on 04.03.2010 from the
  [                  Fact] The facts as stated in the complaint, in brief, are: 2 .1 This complaint was filed by the All India Distillers
  [                  Fact] 2.2 It has been stated that Potable Alcohol Industry of India comprises of Indian M

## 3. Data checks

- every labelled doc must have len(text) == len(labels)
- every label must be in label_vocab.json (exact spelling)
- empty / non-string sentences, line breaks, non-ASCII characters (we cleaned to ASCII in Phase 1)
- duplicate document IDs and texts, inside and across splits


In [4]:
rows, doc_hash = [], collections.defaultdict(list)
for name, docs in SPLITS.items():
    ids = [str(d["id"]) for d in docs]
    sents = [s for d in docs for s in d["text"]]
    labelled = all("labels" in d for d in docs)
    chars = "".join(s for s in sents if isinstance(s, str))
    for d in docs:
        doc_hash[hashlib.md5(" ".join(map(str, d["text"])).encode()).hexdigest()].append(f"{name}:{d['id']}")
    rows.append({
        "split": name, "docs": len(docs), "sentences": len(sents), "labelled": labelled,
        "dup_ids": len(ids) - len(set(ids)),
        "len_mismatch": sum(len(d["text"]) != len(d["labels"]) for d in docs) if labelled else None,
        "unknown_labels": sum(l not in LABEL_SET for d in docs for l in d.get("labels", [])),
        "empty_sents": sum(not str(s).strip() for s in sents),
        "non_str": sum(not isinstance(s, str) for s in sents),
        "with_newline": sum("\n" in s for s in sents if isinstance(s, str)),
        "non_ascii_%": round(100 * sum(ord(c) > 127 for c in chars) / max(len(chars), 1), 3),
    })
checks = pd.DataFrame(rows).set_index("split")
display(checks)
unknown = collections.Counter(l for docs in SPLITS.values() for d in docs for l in d.get("labels", []) if l not in LABEL_SET)
print("unknown labels:", dict(unknown) or "none")
dups = {h: v for h, v in doc_hash.items() if len(v) > 1}
print(f"duplicate document texts: {len(dups)}", list(dups.values())[:10])

,docs,sentences,labelled,dup_ids,len_mismatch,unknown_labels,empty_sents,non_str,with_newline,non_ascii_%
split,,,,,,,,,,
CL_train,40,10593,True,0,0.0,0,0,0,0,0.0
CL_dev,5,1396,True,0,0.0,0,0,0,0,0.0
IT_train,40,6252,True,0,0.0,0,0,0,0,0.0
IT_dev,5,746,True,0,0.0,0,0,0,0,0.0
CL_test,5,1339,False,0,NaN,0,0,0,0,0.0
IT_test,5,858,False,0,NaN,0,0,0,0,0.0


unknown labels: none
duplicate document texts: 0 []


## 4. Label distribution

Macro-F1 averages over labels, so a role with 20 sentences counts as much as Fact with thousands.


In [5]:
lab_splits = [n for n, docs in SPLITS.items() if all("labels" in d for d in docs)]
counts = pd.DataFrame({n: collections.Counter(l for d in SPLITS[n] for l in d["labels"]) for n in lab_splits},
                      index=LABELS).fillna(0).astype(int)
pct = (100 * counts / counts.sum()).round(2).add_suffix(" %")
dist = pd.concat([counts, pct], axis=1)
dist = dist.loc[counts.sum(axis=1).sort_values(ascending=False).index]
display(dist)
docs_with = pd.DataFrame({n: collections.Counter(l for d in SPLITS[n] for l in set(d["labels"])) for n in lab_splits},
                         index=LABELS).fillna(0).astype(int)
print("number of documents containing each label:")
display(docs_with.loc[dist.index])
missing_in_dev = {n: [l for l in LABELS if counts.loc[l, n] == 0] for n in lab_splits}
print("labels with 0 sentences:", missing_in_dev)

,CL_train,CL_dev,IT_train,IT_dev,CL_train %,CL_dev %,IT_train %,IT_dev %
RatioOfTheDecision,2856,288,1445,115,26.96,20.63,23.11,15.42
Fact,2979,249,1119,195,28.12,17.84,17.90,26.14
PrecedentReliedUpon,1305,142,640,112,12.32,10.17,10.24,15.01
ArgumentPetitioner,976,344,637,56,9.21,24.64,10.19,7.51
ArgumentRespondent,996,236,352,36,9.40,16.91,5.63,4.83
Statute,367,58,732,62,3.46,4.15,11.71,8.31
RulingByLowerCourt,438,56,436,46,4.13,4.01,6.97,6.17
PrecedentNotReliedUpon,209,0,360,50,1.97,0.00,5.76,6.70
RulingByPresentCourt,113,18,194,31,1.07,1.29,3.10,4.16
None,53,2,244,31,0.50,0.14,3.90,4.16


number of documents containing each label:


,CL_train,CL_dev,IT_train,IT_dev
RatioOfTheDecision,40,5,40,5
Fact,40,5,38,5
PrecedentReliedUpon,17,2,30,4
ArgumentPetitioner,37,5,35,5
ArgumentRespondent,26,4,33,4
Statute,21,2,29,5
RulingByLowerCourt,16,2,30,3
PrecedentNotReliedUpon,11,0,21,3
RulingByPresentCourt,40,5,40,5
None,11,1,38,5


labels with 0 sentences: {'CL_train': ['PrecedentOverruled'], 'CL_dev': ['Dissent', 'PrecedentNotReliedUpon', 'PrecedentOverruled'], 'IT_train': ['Dissent'], 'IT_dev': ['Dissent', 'PrecedentOverruled']}


## 5. Lengths in words


In [6]:
def pct_row(x):
    x = np.asarray(x)
    return {"mean": round(float(x.mean()), 1), "p50": int(np.percentile(x, 50)), "p90": int(np.percentile(x, 90)),
            "p99": int(np.percentile(x, 99)), "max": int(x.max())}

display(pd.DataFrame({n: pct_row([len(str(s).split()) for d in docs for s in d["text"]]) for n, docs in SPLITS.items()}).T
        .rename_axis("words per sentence"))
display(pd.DataFrame({n: pct_row([len(d["text"]) for d in docs]) for n, docs in SPLITS.items()}).T
        .rename_axis("sentences per document"))

,mean,p50,p90,p99,max
words per sentence,,,,,
CL_train,32.7,27.0,58.0,117.0,733.0
CL_dev,33.7,28.0,57.0,125.0,339.0
IT_train,35.3,28.0,67.0,153.0,1054.0
IT_dev,32.3,26.0,60.0,140.0,287.0
CL_test,33.2,27.0,61.0,137.0,533.0
IT_test,37.6,29.0,72.0,160.0,316.0


,mean,p50,p90,p99,max
sentences per document,,,,,
CL_train,264.8,120.0,760.0,1813.0,1816.0
CL_dev,279.2,184.0,614.0,824.0,848.0
IT_train,156.3,126.0,255.0,430.0,442.0
IT_dev,149.2,120.0,278.0,371.0,382.0
CL_test,267.8,114.0,657.0,913.0,942.0
IT_test,171.6,159.0,326.0,410.0,420.0


## 6. Lengths in bpe32k tokens (cached)

Each sentence is tokenised with a leading space, as it appears inside a document.
This tells us how many neighbouring sentences fit into one 1536-token window.


In [7]:
from tokenizers import Tokenizer
tok = Tokenizer.from_file(str(TOK_PATH))
print("using tokenizer:", TOK_PATH.relative_to(ROOT), "| vocab", tok.get_vocab_size())

cache = OUT / "token_counts.json"
if cache.exists():
    TOKS = read_json(cache)
    print("loaded cached token counts")
else:
    TOKS = {}
    for name, docs in SPLITS.items():
        TOKS[name] = [[len(e.ids) for e in tok.encode_batch([" " + str(s) for s in d["text"]], add_special_tokens=False)]
                      for d in tqdm(docs, desc=f"tokenising {name}")]
    cache.write_text(json.dumps(TOKS))

if TOKS:
    sent_rows, doc_rows = {}, {}
    for name, per_doc in TOKS.items():
        flat = [t for doc in per_doc for t in doc]
        sent_rows[name] = {**pct_row(flat), ">128": sum(t > 128 for t in flat), ">256": sum(t > 256 for t in flat),
                           ">512": sum(t > 512 for t in flat),
                           "sents per 1536 window": round(1536 / float(np.mean(flat)), 1)}
        dtoks = [sum(doc) for doc in per_doc]
        doc_rows[name] = {**pct_row(dtoks), "docs <=1536": sum(t <= 1536 for t in dtoks), "docs": len(dtoks)}
    display(pd.DataFrame(sent_rows).T.rename_axis("bpe32k tokens per sentence"))
    display(pd.DataFrame(doc_rows).T.rename_axis("bpe32k tokens per document"))

using tokenizer: data\tokenizer\bpe32k.json | vocab 32000


tokenising IT_test: 100%|██████████| 5/5 [00:00<00:00, 359.44it/s]


,mean,p50,p90,p99,max,>128,>256,>512,sents per 1536 window
bpe32k tokens per sentence,,,,,,,,,
CL_train,39.2,33.0,69.0,141.0,846.0,139.0,21.0,3.0,39.2
CL_dev,40.6,34.0,69.0,154.0,415.0,24.0,4.0,0.0,37.9
IT_train,42.7,33.0,80.0,188.0,1186.0,172.0,26.0,3.0,36.0
IT_dev,39.1,32.0,72.0,168.0,317.0,19.0,2.0,0.0,39.3
CL_test,39.2,31.0,72.0,162.0,630.0,26.0,5.0,1.0,39.2
IT_test,44.4,35.0,81.0,201.0,371.0,23.0,5.0,0.0,34.6


,mean,p50,p90,p99,max,docs <=1536,docs
bpe32k tokens per document,,,,,,,
CL_train,10380.9,5221.0,26234.0,71035.0,71409.0,4.0,40.0
CL_dev,11322.8,7630.0,24867.0,33559.0,34525.0,1.0,5.0
IT_train,6666.6,5515.0,14638.0,19246.0,19835.0,5.0,40.0
IT_dev,5831.8,4484.0,11344.0,14769.0,15150.0,1.0,5.0
CL_test,10499.6,3602.0,25763.0,35704.0,36809.0,1.0,5.0
IT_test,7627.6,7669.0,13657.0,15932.0,16185.0,0.0,5.0


## 7. Sequence structure (training data)

- same-as-previous rate: how often a sentence has the same role as the one before it (labels come in runs?)
- most common changes between roles
- mean relative position of each role (0 = start of document, 1 = end)


In [8]:
for dom in ["CL", "IT"]:
    docs = SPLITS.get(f"{dom}_train", [])
    if not docs:
        continue
    same = total = 0
    trans, pos = collections.Counter(), collections.defaultdict(list)
    for d in docs:
        labs, n = d["labels"], len(d["labels"])
        for i, l in enumerate(labs):
            pos[l].append(i / max(n - 1, 1))
            if i:
                total += 1
                same += labs[i - 1] == l
                if labs[i - 1] != l:
                    trans[(labs[i - 1], l)] += 1
    print(f"\n===== {dom}_train: same-as-previous {100 * same / total:.1f}% of {total} pairs")
    print("top role changes:", [(f"{a} -> {b}", c) for (a, b), c in trans.most_common(12)])
    print("mean relative position:", {l: round(float(np.mean(p)), 2) for l, p in sorted(pos.items(), key=lambda kv: np.mean(kv[1]))})
    first = collections.Counter(d["labels"][0] for d in docs)
    last = collections.Counter(d["labels"][-1] for d in docs)
    print("first sentence role:", first.most_common(4), "| last sentence role:", last.most_common(4))


===== CL_train: same-as-previous 93.7% of 10553 pairs
top role changes: [('PrecedentReliedUpon -> RatioOfTheDecision', 49), ('Fact -> ArgumentPetitioner', 48), ('RatioOfTheDecision -> PrecedentReliedUpon', 48), ('RatioOfTheDecision -> RulingByPresentCourt', 42), ('ArgumentPetitioner -> Fact', 35), ('Fact -> RatioOfTheDecision', 29), ('Statute -> RatioOfTheDecision', 29), ('RatioOfTheDecision -> Statute', 28), ('Fact -> RulingByLowerCourt', 24), ('ArgumentPetitioner -> RatioOfTheDecision', 19), ('RulingByLowerCourt -> Fact', 19), ('ArgumentPetitioner -> ArgumentRespondent', 19)]
mean relative position: {'Fact': 0.29, 'PrecedentNotReliedUpon': 0.32, 'ArgumentPetitioner': 0.33, 'Issue': 0.35, 'ArgumentRespondent': 0.39, 'Statute': 0.44, 'RulingByLowerCourt': 0.49, 'RatioOfTheDecision': 0.68, 'PrecedentReliedUpon': 0.71, 'None': 0.76, 'Dissent': 0.87, 'RulingByPresentCourt': 0.97}
first sentence role: [('Fact', 36), ('Issue', 2), ('None', 1), ('RatioOfTheDecision', 1)] | last sentence rol

## 8. Repeated sentences

Identical sentences with different labels show how much label noise there is.
Test sentences already seen in training show how much boilerplate is shared.


In [9]:
for dom in ["CL", "IT"]:
    train = SPLITS.get(f"{dom}_train", []) + SPLITS.get(f"{dom}_dev", [])
    seen = collections.defaultdict(collections.Counter)
    for d in train:
        for s, l in zip(d["text"], d["labels"]):
            seen[str(s).strip().lower()][l] += 1
    repeated = {s: c for s, c in seen.items() if sum(c.values()) > 1}
    conflict = {s: c for s, c in repeated.items() if len(c) > 1}
    test = [str(s).strip().lower() for d in SPLITS.get(f"{dom}_test", []) for s in d["text"]]
    print(f"{dom}: {len(repeated)} repeated train+dev sentences, {len(conflict)} with conflicting labels; "
          f"test sentences seen in train+dev: {sum(s in seen for s in test)}/{len(test)}")
    for s, c in list(conflict.items())[:5]:
        print(f"   {dict(c)}  '{s[:80]}'")

CL: 369 repeated train+dev sentences, 207 with conflicting labels; test sentences seen in train+dev: 67/1339
   {'RulingByLowerCourt': 1, 'RatioOfTheDecision': 1}  'further, the alleged conduct of the ops such as failure to adhere to its obligat'
   {'RulingByLowerCourt': 1, 'RatioOfTheDecision': 1}  'the commission finds that the conduct of the ops whereby they have not undertake'
   {'RulingByLowerCourt': 1, 'RatioOfTheDecision': 1}  'thus, in view of foregoing, the commission is of the opinion that the conduct of'
   {'ArgumentRespondent': 1, 'RatioOfTheDecision': 1}  '63 on july 24, 2010, rs. 2511.26 on july 26, 2010 and rs. 2560.15 on july 27, 20'
   {'ArgumentRespondent': 1, 'RatioOfTheDecision': 2}  'it has also provided daily analysis of quantity of sugar sold during the period '
IT: 140 repeated train+dev sentences, 31 with conflicting labels; test sentences seen in train+dev: 3/858
   {'ArgumentPetitioner': 1, 'PrecedentReliedUpon': 1}  'cit and ors.'
   {'ArgumentRespondent'

## 9. Save summary


In [10]:
summary = {
    "labels": LABELS,
    "checks": checks.reset_index().to_dict(orient="records"),
    "label_counts": counts.to_dict(),
    "test_sentences": {n: [len(d["text"]) for d in SPLITS[n]] for n in SPLITS if n.endswith("test")},
    "tokenizer": str(TOK_PATH.relative_to(ROOT)),
}
(OUT / "data_summary.json").write_text(json.dumps(summary, indent=1, default=str))
print("saved", (OUT / "data_summary.json").relative_to(ROOT))

saved experiments\phase3\p3_01_data\data_summary.json
